# vLLM_RT — Real-Time Serving via Driver Proxy

Launches a vLLM OpenAI-compatible API server on the driver node, exposed through the Databricks driver proxy. Provides always-hot, low-latency inference for demos and prototyping.

**Pattern**: Continuous Databricks Job — the `%%sh` cell at the bottom blocks indefinitely, keeping the vLLM server running as long as the job is active.

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | g5.2xlarge or equivalent (1× NVIDIA A10G, 24 GB VRAM) |
| Runtime | `15.4.x-gpu-ml-scala2.12` (MLR 15.4 GPU) |
| Workers | 0 (single-node, `spark.master = local[*, 4]`) |
| Libraries | **None** — all packages installed via `%pip` below |

> **Important**: vLLM must NOT be installed as a cluster library — it causes ABI conflicts with the MLR pre-installed PyTorch.

### Deployment
- Run as a **CONTINUOUS** Databricks Job (`pause_status=UNPAUSED`)
- The `%%sh` cell blocks indefinitely — the job keeps running as long as vLLM is up
- Pause the job when not in use to stop GPU billing

### Prerequisites
- `setup/00_download_model` — model downloaded to Unity Catalog Volume

### Known Gotchas
- **vLLM `/models` returns 200 before the model is fully warm** — the first image inference may 500/crash even though the health check passes. Send a text warmup request before image inference.
- **HTTP 401 from driver proxy** means the cluster is terminated, not an auth error.
- **`--limit-mm-per-prompt image=1`** (not 20) is required to avoid GPU OOM on 24 GB VRAM.

### Install Dependencies

| Package | Why |
|---------|-----|
| `vllm==0.7.3` | GPU inference engine (pinned — 0.8.x crashes on MLR 15.4) |
| `openai>=1.50` | Required by vLLM's OpenAI-compatible API internals |
| `transformers>=4.45,<5` | Model tokenizer/config (v5 removes `all_special_tokens_extended`) |

In [ ]:
%pip install "vllm==0.7.3" "openai>=1.50" "transformers>=4.45,<5"

In [ ]:
dbutils.library.restartPython()

### Configuration and Model Preparation

This cell does three things:
1. **Reads `config.yaml`** for catalog, schema, volume, port, and model name
2. **Creates a local symlink farm** at `/local_disk0/mineru_model/` — because UC Volumes are read-only FUSE mounts, we can't patch files in place
3. **Patches `config.json`** to fix the `rope_scaling` conflict (`rope_type=default` vs `type=mrope`) that vLLM 0.7.3 rejects

The symlink approach is fast and uses minimal disk — only `config.json` is actually copied.

In [ ]:
import yaml, os

# Resolve project root
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

cfg        = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG    = cfg["catalog"]
SCHEMA     = cfg["schema"]
MODEL_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{cfg['volume']}/{cfg['model_subpath']}"
PORT       = str(cfg["vllm_port"])
MODEL_NAME = cfg["vllm_model_name"]

# Export to environment variables — the %%sh cell runs in a subprocess
# and cannot access Python variables directly.
os.environ["VLLM_PORT"]       = PORT
os.environ["VLLM_MODEL_NAME"] = MODEL_NAME

# --- Create local model directory with patched config.json ---
# UC Volumes are read-only FUSE mounts — we cannot modify config.json in place.
# Strategy: symlink every file except config.json to the Volume path,
# then write a patched config.json to local disk.
import json as _json, shutil as _shutil

LOCAL_MODEL = "/local_disk0/mineru_model"
if os.path.exists(LOCAL_MODEL):
    _shutil.rmtree(LOCAL_MODEL)  # Clean up from any previous run
os.makedirs(LOCAL_MODEL)

# Symlink all model files (weights, tokenizer, etc.) — fast, no disk usage
for _item in os.listdir(MODEL_PATH):
    _src = os.path.join(MODEL_PATH, _item)
    _dst = os.path.join(LOCAL_MODEL, _item)
    if _item == "config.json":
        continue  # Will be patched separately
    os.symlink(_src, _dst)

# Patch config.json: reconcile rope_scaling conflict
# Model has rope_type=default and type=mrope — vLLM 0.7.3 rejects this mismatch.
with open(f"{MODEL_PATH}/config.json") as _f:
    _mcfg = _json.load(_f)
_rs = _mcfg.get("rope_scaling", {})
if "rope_type" in _rs and "type" in _rs and _rs["rope_type"] != _rs["type"]:
    _rs["rope_type"] = _rs["type"]
    print(f"Patched rope_scaling: rope_type={_rs['rope_type']}")
with open(f"{LOCAL_MODEL}/config.json", "w") as _f:
    _json.dump(_mcfg, _f, indent=2)

os.environ["VLLM_MODEL_PATH"] = LOCAL_MODEL
print(f"Local model ready at {LOCAL_MODEL} (symlinked + patched config.json)")

### Driver Proxy URL

The Databricks driver proxy exposes HTTP services running on the driver node. The URL format is:
```
https://{workspace}/driver-proxy-api/o/0/{cluster_id}/{port}/v1
```

Clients authenticate with a Databricks PAT token in the `Authorization: Bearer` header.

In [ ]:
# Build the driver proxy URL from cluster metadata
workspace_url = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}'.rstrip("/")
cluster_id    = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
base_url      = f"{workspace_url}/driver-proxy-api/o/0/{cluster_id}/{PORT}/v1"

print(f"vLLM server starting on port {PORT}")
print(f"Driver proxy base URL : {base_url}")
print(f"Models endpoint       : {base_url}/models")
print(f"Chat completions      : {base_url}/chat/completions")
print(f"Cluster ID            : {cluster_id}")
print(f"Model name            : {MODEL_NAME}")

### Launch vLLM Server

This cell **blocks indefinitely** — the vLLM server runs until the job is paused or the cluster is terminated.

| Flag | Purpose |
|------|---------|
| `--host 0.0.0.0` | Listen on all interfaces (required for driver proxy) |
| `--dtype bfloat16` | Half-precision to fit in 24 GB VRAM |
| `--gpu-memory-utilization 0.90` | Reserve 90% of GPU for KV-cache + model weights |
| `--max-model-len 8192` | Maximum context window (tokens) |
| `--limit-mm-per-prompt image=1` | One image per request to avoid OOM |
| `--trust-remote-code` | Required for Qwen2VL model architecture |

In [ ]:
%%sh
python -m vllm.entrypoints.openai.api_server \
  --model         "$VLLM_MODEL_PATH" \
  --host          0.0.0.0 \
  --port          "$VLLM_PORT" \
  --served-model-name "$VLLM_MODEL_NAME" \
  --dtype         bfloat16 \
  --gpu-memory-utilization 0.90 \
  --max-model-len 8192 \
  --limit-mm-per-prompt image=1 \
  --trust-remote-code